In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'pandas', 'matplotlib', 'numpy'])
print('done')


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path('../../').resolve()
sys.path.append(str(ROOT / 'benches'))
import importlib
import sigmod_exp_common as _sigmod_exp_common
importlib.reload(_sigmod_exp_common)
sys.path.append(str(ROOT / 'benches' / 'hash_join' / 'htap_simulation'))

from sigmod_exp_common import (
    TOL,
    SIGMOD_BUCKET_NUM,
    SIGMOD_HTAP_TXN_COUNT,
    SIGMOD_HTAP_WAREHOUSE_COUNT,
    SIGMOD_READABLE_EVERY,
    apply_paper_style,
    display_name,
    ensure_dirs,
    normalize_repair,
    normalize_table,
    run_checked,
)

apply_paper_style(ROOT)

EXP_DIR = (ROOT / 'benches' / 'sigmod_exp1_space_breakdown').resolve()
DATA_DIR = EXP_DIR / 'data'
FIGS_DIR = EXP_DIR / 'figs'
ensure_dirs(DATA_DIR, FIGS_DIR)

BIN = ROOT / 'target' / 'release' / 'htap_wkld'
TABLE_TYPES = ['naive', 'ivmh', 'heap', 'chain', 'par']
SELECTED_SERIES = [
    ('naive', 'NR'),
    ('ivmh', 'NR'),
    ('heap', 'WR'),
    ('chain', 'WR'),
    ('par', 'WR'),
]

CONFIG = {
    'warehouse_count': SIGMOD_HTAP_WAREHOUSE_COUNT,
    'txn_count': SIGMOD_HTAP_TXN_COUNT,
    'bucket_num': SIGMOD_BUCKET_NUM,
    'update_ratio': 0.002,
    'probe_ratio': 0.0001,
    'scan_reuse_ratio': 0.5,
    'analytical_uniform': True,
    'txn_gc_ratio': 0.05,
    'readable_every': SIGMOD_READABLE_EVERY,
    'force_rerun': False,
}

WORKLOADS = {
    'RH': 0.80,
    'B': 0.50,
    'WH': 0.20,
}

RUN_TAG = (
    f"wc{CONFIG['warehouse_count']}_tx{CONFIG['txn_count']}_b{CONFIG['bucket_num']}"
    f"_u{str(CONFIG['update_ratio']).replace('.', 'p')}"
    f"_p{str(CONFIG['probe_ratio']).replace('.', 'p')}"
    f"_re{CONFIG['readable_every']}"
)

BASE_ARGS = [
    '--update-ratio', str(CONFIG['update_ratio']),
    '--probe-ratio', str(CONFIG['probe_ratio']),
    '--txn-count', str(CONFIG['txn_count']),
    '--warehouse-count', str(CONFIG['warehouse_count']),
    '--scan-reuse-ratio', str(CONFIG['scan_reuse_ratio']),
    '--txn-gc-ratio', str(CONFIG['txn_gc_ratio']),
    '--bucket-num', str(CONFIG['bucket_num']),
    '--readable-every', str(CONFIG['readable_every']),
]
if CONFIG['analytical_uniform']:
    BASE_ARGS += ['--analytical-uniform', 'on']

print('ROOT   :', ROOT)
print('BIN    :', BIN)
print('OUTDIR :', DATA_DIR)


In [ ]:
print('Building htap_wkld...')
run_checked(['cargo', 'build', '--release', '--bin', 'htap_wkld'], ROOT)
print('Build OK')


In [ ]:
def label_for(table_type, repair_type):
    if table_type in {'naive', 'ivmh'}:
        return display_name(table_type, '')
    return display_name(table_type, repair_type)


def load_space_csv(path):
    df = pd.read_csv(path, keep_default_na=False)
    df['table_type'] = df['table_type'].map(normalize_table)
    df['repair_type'] = df['repair_type'].map(normalize_repair)
    df['current_mib'] = df['current_space'] / (1024 * 1024)
    df['history_mib'] = df['history_space'] / (1024 * 1024)
    df['metadata_mib'] = df['metadata_space'] / (1024 * 1024)
    df['total_mib'] = df['total_space'] / (1024 * 1024)
    return df


def run_workload(name, analytical_ratio):
    merged_csv = DATA_DIR / f'sigmod_exp1_space_{name}_{RUN_TAG}.csv'
    if merged_csv.exists() and not CONFIG['force_rerun']:
        print(f'Using cached CSV: {merged_csv.name}')
        return load_space_csv(merged_csv)

    dfs = []
    args = BASE_ARGS + ['--analytical-ratio', str(analytical_ratio)]
    for table_type in TABLE_TYPES:
        print(f'  workload={name} table={table_type}')
        table_csv = DATA_DIR / f'sigmod_exp1_space_{name}_{table_type}_{RUN_TAG}.csv'
        if table_csv.exists() and CONFIG['force_rerun']:
            table_csv.unlink()
        if not table_csv.exists():
            run_checked([
                str(BIN),
                *args,
                '--table-type', table_type,
                '--space-stat', str(table_csv),
            ], ROOT, quiet=True)
        dfs.append(load_space_csv(table_csv))

    merged = pd.concat(dfs, ignore_index=True)
    merged.to_csv(merged_csv, index=False)
    return merged


space_frames = {}
for workload_name, analytical_ratio in WORKLOADS.items():
    space_frames[workload_name] = run_workload(workload_name, analytical_ratio)

space_df = pd.concat(
    [df.assign(workload=workload_name) for workload_name, df in space_frames.items()],
    ignore_index=True,
)
space_df['current_mib'] = space_df['current_space'] / (1024 * 1024)
space_df['history_mib'] = space_df['history_space'] / (1024 * 1024)
space_df['metadata_mib'] = space_df['metadata_space'] / (1024 * 1024)
space_df['total_mib'] = space_df['total_space'] / (1024 * 1024)

display_cols = [
    'workload', 'table_type', 'repair_type',
    'current_mib', 'history_mib', 'metadata_mib', 'total_mib'
]
display(space_df[display_cols].round(2))


In [ ]:
COLOR_MAP = {
    'Current': TOL['grey'],
    'History': TOL['blue'],
    'Metadata': TOL['yellow'],
}


def selected_rows(df):
    rows = []
    for table_type, repair_type in SELECTED_SERIES:
        row = df[(df['table_type'] == table_type) & (df['repair_type'] == repair_type)]
        if row.empty:
            raise ValueError(f'Missing row for {(table_type, repair_type)}')
        rows.append(row.iloc[0])
    return pd.DataFrame(rows)


fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.2), sharey=True)
workload_order = ['RH', 'B', 'WH']
stack_order = ['Current', 'History', 'Metadata']

for ax, workload_name in zip(axes, workload_order):
    plot_df = selected_rows(space_frames[workload_name]).copy()
    plot_df['label'] = [label_for(t, r) for t, r in zip(plot_df['table_type'], plot_df['repair_type'])]
    x = np.arange(len(plot_df))
    bottoms = np.zeros(len(plot_df))
    values = {
        'Current': plot_df['current_mib'].to_numpy(),
        'History': plot_df['history_mib'].to_numpy(),
        'Metadata': plot_df['metadata_mib'].to_numpy(),
    }
    for key in stack_order:
        ax.bar(x, values[key], bottom=bottoms, width=0.62, color=COLOR_MAP[key], edgecolor='black', linewidth=0.5, label=key)
        bottoms += values[key]
    for xi, total in zip(x, plot_df['total_mib'].to_numpy()):
        ax.text(xi, total + 0.03 * max(1.0, plot_df['total_mib'].max()), f'{total:.1f}', ha='center', va='bottom', fontsize=8)
    ax.set_title(workload_name)
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df['label'], rotation=0)
    ax.yaxis.grid(True, linestyle='--', linewidth=0.6, alpha=0.6)

axes[0].set_ylabel('Derived-State Memory (MiB)')
handles = [plt.Rectangle((0, 0), 1, 1, facecolor=COLOR_MAP[key], edgecolor='black', linewidth=0.5) for key in stack_order]
fig.legend(handles, stack_order, ncol=3, loc='upper center', frameon=True, bbox_to_anchor=(0.5, 1.02))
fig.tight_layout(rect=[0, 0, 1, 0.93])

pdf_path = FIGS_DIR / 'sigmod_exp1_space_breakdown.pdf'
png_path = FIGS_DIR / 'sigmod_exp1_space_breakdown.png'
fig.savefig(pdf_path, bbox_inches='tight')
fig.savefig(png_path, dpi=200, bbox_inches='tight')
plt.show()
print('saved:', pdf_path)
print('saved:', png_path)


In [ ]:
rh_df = selected_rows(space_frames['RH']).copy()
rh_df['label'] = [label_for(t, r) for t, r in zip(rh_df['table_type'], rh_df['repair_type'])]

fig, ax = plt.subplots(1, 1, figsize=(6.4, 4.2))
x = np.arange(len(rh_df))
bottoms = np.zeros(len(rh_df))
values = {
    'Current': rh_df['current_mib'].to_numpy(),
    'History': rh_df['history_mib'].to_numpy(),
    'Metadata': rh_df['metadata_mib'].to_numpy(),
}
for key in stack_order:
    ax.bar(x, values[key], bottom=bottoms, width=0.62, color=COLOR_MAP[key], edgecolor='black', linewidth=0.5, label=key)
    bottoms += values[key]

y_max = float(rh_df['total_mib'].max())
for xi, total in zip(x, rh_df['total_mib'].to_numpy()):
    ax.text(xi, total + 0.02 * y_max, f'{total:.1f}', ha='center', va='bottom', fontsize=8)

ax.set_title('RH')
ax.set_ylabel('Derived-State Memory (MiB)')
ax.set_xticks(x)
ax.set_xticklabels(rh_df['label'], rotation=0)
ax.set_ylim(0, y_max * 1.12)
ax.yaxis.grid(True, linestyle='--', linewidth=0.6, alpha=0.6)
ax.legend(ncol=3, loc='upper center', frameon=True, bbox_to_anchor=(0.5, 1.14))
fig.tight_layout()

pdf_path = FIGS_DIR / 'sigmod_exp1_space_breakdown_rh.pdf'
png_path = FIGS_DIR / 'sigmod_exp1_space_breakdown_rh.png'
fig.savefig(pdf_path, bbox_inches='tight')
fig.savefig(png_path, dpi=200, bbox_inches='tight')
plt.show()
print('saved:', pdf_path)
print('saved:', png_path)
